In [ ]:
import pandas as pd
import numpy as np
import os
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
from itertools import product  # For creating parameter grid
import warnings
warnings.filterwarnings("ignore")

# Load data
csv_path = 'all_temperature_cleaned.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
df['datetime'] = pd.to_datetime(df['timestamp'] + ' ' + df['time'], format='%Y-%m-%d %H:%M')

regions = ['Rakhiyal', 'Bopal', 'Ambawadi', 'Chandkheda', 'Vastral']

# Split data into train (2019-2022), validation (2023), test (2024)
train_df = df[(df['datetime'].dt.year >= 2019) & (df['datetime'].dt.year <= 2022)].copy()
valid_df = df[df['datetime'].dt.year == 2023].copy()
test_df = df[df['datetime'].dt.year == 2024].copy()

# Prepare output folders
output_folder_2024 = 'trial/predictions_2024'
output_folder_2025 = 'predictions_2025'
os.makedirs(output_folder_2024, exist_ok=True)
os.makedirs(output_folder_2025, exist_ok=True)

current_dir = os.getcwd()
metrics_list = []

# Define parameter grids
p_values = [0, 1, 2]
d_values = [0, 1]
q_values = [0, 1, 2]
P_values = [0, 1]
D_values = [0, 1]
Q_values = [0, 1]
m = 24  # Seasonal period for hourly data (daily seasonality)

# Create a list of all parameter combinations
param_grid = list(product(p_values, d_values, q_values, P_values, D_values, Q_values))

for region in regions:
    print(f"\nTuning SARIMA parameters for region: {region}")

    train_series = train_df.set_index('datetime')[region].interpolate(method='time')
    valid_series = valid_df.set_index('datetime')[region].interpolate(method='time')
    test_series = test_df.set_index('datetime')[region].interpolate(method='time')

    best_rmse = float('inf')
    best_params = None
    best_model = None

    # Grid search over parameters
    for params in param_grid:
        order = (params[0], params[1], params[2])
        seasonal_order = (params[3], params[4], params[5], m)
        try:
            model = SARIMAX(train_series, order=order, seasonal_order=seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            model_fit = model.fit(disp=False)

            # Forecast on validation set
            predictions = model_fit.forecast(steps=len(valid_series))

            rmse = np.sqrt(mean_squared_error(valid_series, predictions))

            if rmse < best_rmse:
                best_rmse = rmse
                best_params = {'order': order, 'seasonal_order': seasonal_order}
                best_model = model_fit  # Store the fitted model

        except Exception as e:
            print(f"Skipping parameters {params} due to error: {e}")
            continue

    print(f"Best params for {region}: {best_params} with validation RMSE: {best_rmse:.4f}")

    # Fit final model on train + validation data
    combined_series = pd.concat([train_series, valid_series])
    final_model = SARIMAX(combined_series, order=best_params['order'], seasonal_order=best_params['seasonal_order'],
                        enforce_stationarity=False, enforce_invertibility=False)
    final_model_fit = final_model.fit(disp=False)

    # Forecast 2024 (test)
    forecast = final_model_fit.forecast(steps=len(test_series))

    # Calculate metrics on 2024 test data
    mse = mean_squared_error(test_series, forecast)
    rmse = np.sqrt(mse)
    r2 = 1 - np.sum((test_series - forecast) ** 2) / np.sum((test_series - np.mean(test_series)) ** 2)

    print(f"  2024 MSE: {mse:.4f}")
    print(f"  2024 RMSE: {rmse:.4f}")
    print(f"  2024 R2 Score: {r2:.4f}")

    # Prepare 2024 prediction DataFrame
    pred_dates = test_series.index
    pred_2024_df = pd.DataFrame({
        'date': pred_dates.date,
        'hour': pred_dates.hour,
        'predicted_temperature': forecast,
        'actual_temperature': test_series.values
    })

    filename_2024 = f"sarimax_{region.lower()}_2024.csv"
    pred_2024_df.to_csv(os.path.join(current_dir, filename_2024), index=False)
    print(f"Saved 2024 SARIMAX predictions for {region} as {filename_2024}")

    # Forecast 2025 hourly iteratively
    date_range_2025 = pd.date_range(start='2025-01-01 00:00', end='2025-12-31 23:00', freq='H')
    n_hours_2025 = len(date_range_2025)

    forecast_2025 = final_model_fit.forecast(steps=n_hours_2025)

    pred_2025_df = pd.DataFrame({
        'date': date_range_2025.date,
        'hour': date_range_2025.hour,
        'predicted_temperature': forecast_2025
    })

    filename_2025 = f"sarimax_{region.lower()}_2025.csv"
    pred_2025_df.to_csv(os.path.join(output_folder_2025, filename_2025), index=False)
    print(f"Saved 2025 SARIMAX predictions for {region} as {os.path.join(output_folder_2025, filename_2025)}")

    # Store metrics
    metrics_list.append({
        'region': region,
        'mse_2024': mse,
        'rmse_2024': rmse,
        'r2_2024': r2,
        'best_params': best_params
    })

# Save all metrics to CSV
metrics_df = pd.DataFrame(metrics_list)
metrics_filename = 'sarimax_auto_tuned_metrics_2024.csv'
metrics_df.to_csv(os.path.join(current_dir, metrics_filename), index=False)
print(f"Saved auto-tuned SARIMAX metrics for all regions as {metrics_filename}")



Tuning SARIMA parameters for region: Rakhiyal
Skipping parameters (0, 0, 0, 1, 1, 0) due to error: Unable to allocate 616. MiB for an array with shape (48, 48, 35064) and data type float64
Skipping parameters (0, 0, 0, 1, 1, 1) due to error: Unable to allocate 642. MiB for an array with shape (49, 49, 35064) and data type float64
Skipping parameters (0, 0, 1, 0, 1, 1) due to error: Unable to allocate 669. MiB for an array with shape (50, 50, 35065) and data type float64
Skipping parameters (0, 0, 1, 1, 1, 0) due to error: Unable to allocate 616. MiB for an array with shape (48, 48, 35065) and data type float64
Skipping parameters (0, 0, 1, 1, 1, 1) due to error: Unable to allocate 669. MiB for an array with shape (50, 50, 35065) and data type float64
Skipping parameters (0, 0, 2, 0, 1, 0) due to error: Unable to allocate 195. MiB for an array with shape (35064, 27, 27) and data type float64
Skipping parameters (0, 0, 2, 0, 1, 1) due to error: Unable to allocate 13.6 MiB for an array w